## Modeling tumor growth with the Dixon model and optimizing immunotherapy dosing with a Genetic Algorithm


This notebook is where the Dixon model can be run and saved. Its similar to the KP model notebook but focuses on the Dixon model specifically.



**This notebook uses the Dixon model.**

Genetic algorithm:
DOI: https://doi-org.proxy.lib.ohio-state.edu/10.1103/PhysRevE.70.016211

Onco model:
DOI: https://doi.org/10.1371/journal.pone.0004271

Dixon et al.:
DOI: https://doi-org.proxy.lib.ohio-state.edu/10.1016/j.mbs.2024.109187

Sarv Ahrabi and Momenzadeh:
DOI: https://doi.org/10.1007/s00285-020-01525-7



In [1]:
# Main module
%reset -f

# autoreload imports
%load_ext autoreload
%autoreload 2

# import required libraries
import os
import itertools
import csv
import time
from pathlib import Path
try:
    import tqdm
except ImportError:
    from types import SimpleNamespace
    tqdm = SimpleNamespace(tqdm=lambda values, **kwargs: values)

# Import scientific libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy.integrate import solve_ivp
import random
import pygad

# import model functions and Genetic Algorithm
from Model.Dixon_model import dixon_dI_dt, dixon_dC1_dt, dixon_dC2_dt, dixon_dT_dt, dixon_coupled
from GA.fitness_function_Dixon import GeneticAlgorithmDixon
from Model.integration import dixon_integrate


In [2]:
# Set integration method for Dixon model
integration_method = "Radau"  # Stable for long runs; "rk4" and "RK45" also supported.
log_cell_integration = True  # Retain very small populations between time steps.


## Model with no treatment input


In [3]:
# Non-dimensional parameters
# Arrow indicates parameters that should be varied for different simulations

# Carrying capacity for tumor cells
# This matches the b=1e-9 used in the KP model for inverse tumor carrying capacity.
K = 1e9

# eq.1
beta = 15.67*1e-8 # production rate of IL-2 by CD4+ cells range=[0, 5.556]
mu_0 = 55.56 # decay rate of IL-2 range=[0, 55.56]
s = 0 # treatment term or external source of IL-2

# eq.2
k1 = 3.43 # stimulation rate of CD4+ cells by tumor cells range=[0.167, 611.11]
mu_1 = 0.93 # death rate of CD4+ cells range=[0.167, 1.833]

# eq.3
k2 = 1.67 # stimulation rate of CD8+ and NK cells by tumor cells range=[0.167, 611.11]
mu_2 = 0.93 # death rate of CD8+ and NK cells range=[0.056, 5.167]
sigma = 0.692 # maximum proliferation rate of CD8+ by IL-2 range=[0.689, 16.51]
g = 0.02 # half-saturation constant for IL-2 stimulation range=[1e-6, 2e-2]

# eq.4
alpha = 715.72 # killing rate of tumor cells by CD8+ and NK cells range=[1.889, 1667]

# Simulation parameters
num_steps = 2000 # number of time steps <---
total_time = 4000 # total time (days) <---

# Time arrays
lambda_tumor = 0.18 # tumor growth rate setting the physical time scale (days^-1)
t_s = lambda_tumor # Independent of the observation duration.
t = np.linspace(0, total_time, num_steps)
tau = t * t_s

# Cell populations and cytokine concentrations
C1 = np.zeros(num_steps)
C2 = np.zeros(num_steps)
I = np.zeros(num_steps)
T = np.zeros(num_steps)

# Dimensionalized cell populations and cytokine concentrations
C1_bar = np.zeros(num_steps)
C2_bar = np.zeros(num_steps)
I_bar = np.zeros(num_steps)
T_bar = np.zeros(num_steps)

# Initial conditions
C1_bar[0] = 1e5 # CD4+ cells
C2_bar[0] = 1e5 # CD8+ and NK cells
I_bar[0] = 1e2 # IL-2
T_bar[0] = 1e5 # tumor cells

# Non-dimensionalize
C1[0] = C1_bar[0] / K
C2[0] = C2_bar[0] / K
I[0] = I_bar[0] / K
T[0] = T_bar[0] / K

# External therapy parameter
s_array = np.zeros(num_steps)
s_array[0] = s

print("Nondimensionalized parameters:")
print(C1[0], C2[0], I[0], s_array[0], T[0], beta, mu_0,
      k1, mu_1, k2, mu_2, sigma, g, alpha)

# Arrays to hold fitness values
fitness = np.zeros(num_steps)
fitness_history = []


Nondimensionalized parameters:
0.0001 0.0001 1e-07 0.0 0.0001 1.567e-07 55.56 3.43 0.93 1.67 0.93 0.692 0.02 715.72


In [4]:
# Initialize Dixon integrator
integrator = dixon_integrate()
if log_cell_integration:
    log_state = integrator.to_log_state([C1[0], C2[0], I[0], T[0]])

# Time-stepping loop
for step in range(1, num_steps):
    # External input parameters
    s_array[step] = 0

    # Parameters for time step
    params = [s_array[step], beta, mu_0, k1, mu_1,
              k2, mu_2, sigma, g, alpha]

    # Solve Dixon ODEs for time step
    state = log_state if log_cell_integration else [C1[step-1], C2[step-1], I[step-1], T[step-1]]
    next_state = integrator.integrate(
        state, params,
        t_span=(tau[step-1], tau[step]),
        method=integration_method,
        log_cells=log_cell_integration,
    )
    if log_cell_integration:
        log_state = next_state
        next_state = integrator.from_log_state(log_state)
    C1[step], C2[step], I[step], T[step] = next_state

    # Undo the K scaling used for every initial condition.
    C1_bar[step] = C1[step] * K
    C2_bar[step] = C2[step] * K
    I_bar[step] = I[step] * K
    T_bar[step] = T[step] * K

# Save simulation data to CSV file
with open("Output_data/simu_data_test1.csv", mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['t', 'tau', 'C1', 'C2', 'I', 'T', 'C1_bar', 'C2_bar', 'I_bar', 'T_bar', 's', 'alpha', 'Fitness'])
    for i in range(num_steps):
        writer.writerow([
            t[i], tau[i], C1[i], C2[i], I[i], T[i],
            C1_bar[i], C2_bar[i], I_bar[i], T_bar[i], s_array[i], alpha,
            fitness_history[i-1] if 0 < i <= len(fitness_history) else '',
        ])

print("Simulation completed.")


Simulation completed.


**Sweep through array of alpha values.**

s = 0

In [6]:
# Different alpha values for each run
alpha_values = np.concatenate(([1.889, 715.72, 1667], np.linspace(1, 1700, 200)))
start_time = time.time()

# Resume settings
overwrite_existing = False
resume_from_index = 0
output_dir = Path("Output_data/dixon_alpha_sweep_no_treatment")
output_dir.mkdir(parents=True, exist_ok=True)

for alpha_s in tqdm.tqdm(alpha_values[resume_from_index:], desc="No-treatment alpha sweep"):
    # Non-dimensional parameters
    # Arrow indicates parameters that should be varied for different simulations

    # Carrying capacity for tumor cells
    # This matches the b=1e-9 used in the KP model for inverse tumor carrying capacity.
    K = 1e9

    # eq.1
    beta = 15.67*1e-8 # production rate of IL-2 by CD4+ cells range=[0, 5.556]
    mu_0 = 55.56 # decay rate of IL-2 range=[0, 55.56]
    s = 0 # treatment term or external source of IL-2

    # eq.2
    k1 = 3.43 # stimulation rate of CD4+ cells by tumor cells range=[0.167, 611.11]
    mu_1 = 0.93 # death rate of CD4+ cells range=[0.167, 1.833]

    # eq.3
    k2 = 1.67 # stimulation rate of CD8+ and NK cells by tumor cells range=[0.167, 611.11]
    mu_2 = 0.93 # death rate of CD8+ and NK cells range=[0.056, 5.167]
    sigma = 0.692 # maximum proliferation rate of CD8+ by IL-2 range=[0.689, 16.51]
    g = 0.02 # half-saturation constant for IL-2 stimulation range=[1e-6, 2e-2]

    # eq.4
    alpha = alpha_s # killing rate of tumor cells by CD8+ and NK cells range=[1.889, 1667]

    # Simulation parameters
    num_steps = 2000 # number of time steps <---
    total_time = 4000 # total time (days) <---

    # Time arrays
    lambda_tumor = 0.18 # tumor growth rate setting the physical time scale (days^-1)
    t_s = lambda_tumor # Independent of the observation duration.
    t = np.linspace(0, total_time, num_steps)
    tau = t * t_s

    # Cell populations and cytokine concentrations
    C1 = np.zeros(num_steps)
    C2 = np.zeros(num_steps)
    I = np.zeros(num_steps)
    T = np.zeros(num_steps)

    # Dimensionalized cell populations and cytokine concentrations
    C1_bar = np.zeros(num_steps)
    C2_bar = np.zeros(num_steps)
    I_bar = np.zeros(num_steps)
    T_bar = np.zeros(num_steps)

    # Initial conditions
    C1_bar[0] = 1e5 # CD4+ cells
    C2_bar[0] = 1e5 # CD8+ and NK cells
    I_bar[0] = 1e2 # IL-2
    T_bar[0] = 1e5 # tumor cells

    # Non-dimensionalize
    C1[0] = C1_bar[0] / K
    C2[0] = C2_bar[0] / K
    I[0] = I_bar[0] / K
    T[0] = T_bar[0] / K

    # External therapy parameter
    s_array = np.zeros(num_steps)
    s_array[0] = s

    # Arrays to hold fitness values
    fitness = np.zeros(num_steps)
    fitness_history = []

    output_path = output_dir / f"simu_data_dixon_alpha_{alpha_s:.17g}.csv"
    temp_output_path = output_path.with_suffix('.csv.tmp')
    columns = ['t', 'tau', 'C1', 'C2', 'I', 'T', 'C1_bar', 'C2_bar', 'I_bar', 'T_bar', 's', 'alpha', 'Fitness']

    if output_path.exists() and not overwrite_existing:
        with output_path.open(newline='') as existing_file:
            reader = csv.reader(existing_file)
            header = next(reader, None)
            row_count = sum(1 for _ in reader)
        if set(columns).issubset(header or []) and row_count == num_steps:
            existing = pd.read_csv(output_path)
            compatible = (
                np.allclose(existing['t'], t)
                and np.allclose(existing['tau'], tau)
                and np.allclose(existing.loc[0, ['C1', 'C2', 'I', 'T']],
                                [C1[0], C2[0], I[0], T[0]])
            )
            # If the file records coefficients, check those as well.
            for name in ['beta', 'mu_0', 'k1', 'mu_1', 'k2', 'mu_2', 'sigma', 'g', 'alpha']:
                if name in existing:
                    compatible = compatible and np.allclose(existing[name], locals()[name])
            if compatible:
                print("Skipping existing result:", output_path.name)
                continue
        raise ValueError(f"{output_path} uses different settings or is incomplete. "
                         "Choose another output_dir or set overwrite_existing=True to regenerate it.")

    # Initialize Dixon integrator
    integrator = dixon_integrate()
    if log_cell_integration:
        log_state = integrator.to_log_state([C1[0], C2[0], I[0], T[0]])

    # Time-stepping loop
    for step in range(1, num_steps):
        # External input parameters
        s_array[step] = 0

        # Parameters for time step
        params = [s_array[step], beta, mu_0, k1, mu_1,
                  k2, mu_2, sigma, g, alpha]

        # Solve Dixon ODEs for time step
        state = log_state if log_cell_integration else [C1[step-1], C2[step-1], I[step-1], T[step-1]]
        next_state = integrator.integrate(
            state, params,
            t_span=(tau[step-1], tau[step]),
            method=integration_method,
            log_cells=log_cell_integration,
        )
        if log_cell_integration:
            log_state = next_state
            next_state = integrator.from_log_state(log_state)
        C1[step], C2[step], I[step], T[step] = next_state

        # Undo the K scaling used for every initial condition.
        C1_bar[step] = C1[step] * K
        C2_bar[step] = C2[step] * K
        I_bar[step] = I[step] * K
        T_bar[step] = T[step] * K

    with temp_output_path.open(mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(columns)
        for i in range(num_steps):
            writer.writerow([
                t[i], tau[i], C1[i], C2[i], I[i], T[i],
                C1_bar[i], C2_bar[i], I_bar[i], T_bar[i], s_array[i], alpha,
                fitness_history[i-1] if 0 < i <= len(fitness_history) else '',
            ])
    temp_output_path.replace(output_path)
    print("Saved as", output_path)

print("Simulation completed.")
end_time = time.time()
print(f"Total simulation time: {(end_time - start_time)/60} minutes")


No-treatment alpha sweep:   0%|          | 1/203 [00:01<05:51,  1.74s/it]

Saved as Output_data/dixon_alpha_sweep_no_treatment/simu_data_dixon_alpha_1.889.csv


No-treatment alpha sweep:   1%|          | 2/203 [00:03<05:58,  1.78s/it]

Saved as Output_data/dixon_alpha_sweep_no_treatment/simu_data_dixon_alpha_715.72000000000003.csv


No-treatment alpha sweep:   1%|▏         | 3/203 [00:05<05:48,  1.74s/it]

Saved as Output_data/dixon_alpha_sweep_no_treatment/simu_data_dixon_alpha_1667.csv


... (396 outputs left out)

No-treatment alpha sweep: 100%|█████████▉| 202/203 [06:03<00:01,  1.79s/it]

Saved as Output_data/dixon_alpha_sweep_no_treatment/simu_data_dixon_alpha_1691.4623115577888.csv


No-treatment alpha sweep: 100%|██████████| 203/203 [06:05<00:00,  1.80s/it]

Saved as Output_data/dixon_alpha_sweep_no_treatment/simu_data_dixon_alpha_1700.csv
Simulation completed.
Total simulation time: 6.089844600359599 minutes


## Impliment genetic algorithm

Note that this simulation will take more time than without the GA, as the GA uses more computational power to run its optimization algorithm.

In [7]:
# Carrying capacity for tumor cells
# This matches the b=1e-9 used in the KP model for inverse tumor carrying capacity.
K = 1e9

# eq.1
beta = 15.67*1e-8 # production rate of IL-2 by CD4+ cells range=[0, 5.556]
mu_0 = 55.56 # decay rate of IL-2 range=[0, 55.56]
s = 0 # treatment term or external source of IL-2

# eq.2
k1 = 3.43 # stimulation rate of CD4+ cells by tumor cells range=[0.167, 611.11]
mu_1 = 0.93 # death rate of CD4+ cells range=[0.167, 1.833]

# eq.3
k2 = 1.67 # stimulation rate of CD8+ and NK cells by tumor cells range=[0.167, 611.11]
mu_2 = 0.93 # death rate of CD8+ and NK cells range=[0.056, 5.167]
sigma = 0.692 # maximum proliferation rate of CD8+ by IL-2 range=[0.689, 16.51]
g = 0.02 # half-saturation constant for IL-2 stimulation range=[1e-6, 2e-2]

# eq.4
alpha = 715.72 # killing rate of tumor cells by CD8+ and NK cells range=[1.889, 1667]

# Simulation parameters
num_steps = 2000 # number of time steps <---
total_time = 4000 # total time (days) <---

# Time arrays
lambda_tumor = 0.18 # tumor growth rate setting the physical time scale (days^-1)
t_s = lambda_tumor # Independent of the observation duration.
t = np.linspace(0, total_time, num_steps)
tau = t * t_s

# Cell populations and cytokine concentrations
C1 = np.zeros(num_steps)
C2 = np.zeros(num_steps)
I = np.zeros(num_steps)
T = np.zeros(num_steps)

# Dimensionalized cell populations and cytokine concentrations
C1_bar = np.zeros(num_steps)
C2_bar = np.zeros(num_steps)
I_bar = np.zeros(num_steps)
T_bar = np.zeros(num_steps)

# Initial conditions
C1_bar[0] = 1e5 # CD4+ cells
C2_bar[0] = 1e5 # CD8+ and NK cells
I_bar[0] = 1e2 # IL-2
T_bar[0] = 1e5 # tumor cells

# Non-dimensionalize
C1[0] = C1_bar[0] / K
C2[0] = C2_bar[0] / K
I[0] = I_bar[0] / K
T[0] = T_bar[0] / K

# External therapy parameter
s_array = np.zeros(num_steps)
s_array[0] = s

print("Nondimensionalized parameters:")
print(C1[0], C2[0], I[0], s_array[0], T[0], beta, mu_0,
      k1, mu_1, k2, mu_2, sigma, g, alpha)

# Arrays to hold fitness values
fitness = np.zeros(num_steps)
fitness_history = []


Nondimensionalized parameters:
0.0001 0.0001 1e-07 0.0 0.0001 1.567e-07 55.56 3.43 0.93 1.67 0.93 0.692 0.02 715.72


In [8]:
# Genetic Algorithm setup
from GA.native_fitness_dixon import native_available

# Arrays to hold fitness values
fitness = np.zeros(num_steps)
fitness_history = []

# Create instance of GeneticAlgorithmDixon 
ga_func = GeneticAlgorithmDixon()

# Create an instance of the GA class
ga_instance = pygad.GA(
    num_generations=30,
    num_parents_mating=10,
    sol_per_pop=100,
    num_genes=5,
    fitness_func=GeneticAlgorithmDixon.fitness_func_batch,
    fitness_batch_size=100,  # Evaluate Dixon candidates as a batch.
    on_generation=GeneticAlgorithmDixon.on_generation,
    keep_parents=-1,
    save_solutions=False,
    init_range_low=0.0,
    init_range_high=1.0,
    gene_space=[{'low': -10.0, 'high': 10.0}]*5,
    mutation_num_genes=2)

print("Genetic Algorithm initialized. Fitness backend:",
      "native" if native_available() else "Python (build GA.native_fitness_dixon for faster runs)")

Genetic Algorithm initialized. Fitness backend: native


In [9]:
# Use the same Dixon coefficients in fitness prediction and integration.
model_params = {
    'beta': beta, 'mu_0': mu_0, 'k1': k1, 'mu_1': mu_1,
    'k2': k2, 'mu_2': mu_2, 'sigma': sigma, 'g': g, 'alpha': alpha,
}
integrator = dixon_integrate()
if log_cell_integration:
    log_state = integrator.to_log_state([C1[0], C2[0], I[0], T[0]])

for step in tqdm.tqdm(range(1, num_steps)):
    # Optimize the single IL-2 input for the current Dixon state.
    ga_instance.environment = {
        't': tau[step-1],
        'C1': C1[step-1], 'C2': C2[step-1],
        'I': I[step-1], 'T': T[step-1],
        'model_params': model_params,
        't_step': tau[step] - tau[step-1],
    }
    # Re-evaluate retained candidates when the environment changes.
    ga_instance.last_generation_parents = None
    ga_instance.last_generation_elitism = None
    ga_instance.run()
    best_solution, best_solution_fitness, _ = ga_instance.best_solution(
        pop_fitness=ga_instance.last_generation_fitness)
    fitness_history.append(best_solution_fitness)

    # Five genes ordered [C1, C2, I, T, 1]; clip negative treatment to zero.
    s_array[step] = GeneticAlgorithmDixon.solution_to_inputs(
        best_solution, C1[step-1], C2[step-1], I[step-1], T[step-1])

    params = [s_array[step]] + [model_params[key] for key in
        ('beta', 'mu_0', 'k1', 'mu_1', 'k2', 'mu_2', 'sigma', 'g', 'alpha')]

    # Advance the four Dixon states with this treatment held constant.
    state = log_state if log_cell_integration else [C1[step-1], C2[step-1], I[step-1], T[step-1]]
    next_state = integrator.integrate(
        state, params,
        t_span=(tau[step-1], tau[step]),
        method=integration_method,
        log_cells=log_cell_integration,
    )
    if log_cell_integration:
        log_state = next_state
        next_state = integrator.from_log_state(log_state)
    C1[step], C2[step], I[step], T[step] = next_state

    C1_bar[step] = C1[step] * K
    C2_bar[step] = C2[step] * K
    I_bar[step] = I[step] * K
    T_bar[step] = T[step] * K

# Restore the physical IL-2 source rate (IU/day).
s_bar = s_array * K * t_s

# Save Dixon states and the single treatment input.
with open("Output_data/simu_data_dixon_ga1.csv", mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['t', 'tau', 'C1', 'C2', 'I', 'T',
                     'C1_bar', 'C2_bar', 'I_bar', 'T_bar', 's', 's_bar', 'alpha', 'Fitness'])
    for i in range(num_steps):
        writer.writerow([
            t[i], tau[i], C1[i], C2[i], I[i], T[i],
            C1_bar[i], C2_bar[i], I_bar[i], T_bar[i],
            s_array[i], s_bar[i], alpha,
            fitness_history[i-1] if 0 < i <= len(fitness_history) else '',
        ])

print("Simulation completed.")


  0%|          | 0/1999 [00:00<?, ?it/s]

100%|██████████| 1999/1999 [00:54<00:00, 36.61it/s]

Simulation completed.


## Dixon alpha sweep with GA input

Sweep tumor-killing strength `alpha` over the same values as the no-treatment sweep. Each run optimizes one nonnegative IL-2 input `s` with five genes.


In [10]:
# Sweep Dixon tumor-killing strength, matching the no-treatment alpha sweep.
# Dixon has no KP antigenicity parameter c.
alpha_values = np.concatenate(([1.889, 715.72, 1667], np.linspace(1, 1700, 200)))
from GA.native_fitness_dixon import native_available
print("Dixon fitness backend:", "native" if native_available() else
      "Python (build GA.native_fitness_dixon for faster runs)")
start_time = time.time()

# Resume settings
from pathlib import Path
overwrite_existing = False  # Set True to regenerate existing outputs.
resume_from_index = 0
output_dir = Path("Output_data/dixon_alpha_sweep_ga_v2")
output_dir.mkdir(parents=True, exist_ok=True)

for alpha_s in tqdm.tqdm(alpha_values[resume_from_index:], desc="Dixon GA alpha sweep"):
    # Carrying capacity for tumor cells
    # This matches the b=1e-9 used in the KP model for inverse tumor carrying capacity.
    K = 1e9

    # eq.1
    beta = 15.67*1e-8 # production rate of IL-2 by CD4+ cells range=[0, 5.556]
    mu_0 = 55.56 # decay rate of IL-2 range=[0, 55.56]
    s = 0 # treatment term or external source of IL-2

    # eq.2
    k1 = 3.43 # stimulation rate of CD4+ cells by tumor cells range=[0.167, 611.11]
    mu_1 = 0.93 # death rate of CD4+ cells range=[0.167, 1.833]

    # eq.3
    k2 = 1.67 # stimulation rate of CD8+ and NK cells by tumor cells range=[0.167, 611.11]
    mu_2 = 0.93 # death rate of CD8+ and NK cells range=[0.056, 5.167]
    sigma = 0.692 # maximum proliferation rate of CD8+ by IL-2 range=[0.689, 16.51]
    g = 0.02 # half-saturation constant for IL-2 stimulation range=[1e-6, 2e-2]

    # eq.4
    alpha = float(alpha_s) # killing rate of tumor cells by CD8+ and NK cells range=[1.889, 1667]

    # Simulation parameters
    num_steps = 2000 # number of time steps <---
    total_time = 4000 # total time (days) <---

    # Time arrays
    lambda_tumor = 0.18 # tumor growth rate setting the physical time scale (days^-1)
    t_s = lambda_tumor # Independent of the observation duration.
    t = np.linspace(0, total_time, num_steps)
    tau = t * t_s

    # Cell populations and cytokine concentrations
    C1 = np.zeros(num_steps)
    C2 = np.zeros(num_steps)
    I = np.zeros(num_steps)
    T = np.zeros(num_steps)

    # Dimensionalized cell populations and cytokine concentrations
    C1_bar = np.zeros(num_steps)
    C2_bar = np.zeros(num_steps)
    I_bar = np.zeros(num_steps)
    T_bar = np.zeros(num_steps)

    # Initial conditions
    C1_bar[0] = 1e5 # CD4+ cells
    C2_bar[0] = 1e5 # CD8+ and NK cells
    I_bar[0] = 1e2 # IL-2
    T_bar[0] = 1e5 # tumor cells

    # Non-dimensionalize
    C1[0] = C1_bar[0] / K
    C2[0] = C2_bar[0] / K
    I[0] = I_bar[0] / K
    T[0] = T_bar[0] / K

    # External therapy parameter
    s_array = np.zeros(num_steps)
    s_array[0] = s

    # Arrays to hold fitness values
    fitness = np.zeros(num_steps)
    fitness_history = []

    # Use the same Dixon coefficients in fitness prediction and integration.
    model_params = {
        'beta': beta, 'mu_0': mu_0, 'k1': k1, 'mu_1': mu_1,
        'k2': k2, 'mu_2': mu_2, 'sigma': sigma, 'g': g, 'alpha': alpha,
    }

    output_path = output_dir / f"simu_data_dixon_alpha_ga_{alpha:.17g}.csv"
    temp_output_path = output_path.with_suffix('.csv.tmp')
    metadata = dict(model_params, K=K, lambda_tumor=lambda_tumor,
                    integration_method=integration_method, log_cell_integration=log_cell_integration,
                    controller_version='dixon_continuous_genes_v2')
    columns = ['t', 'tau', 'C1', 'C2', 'I', 'T',
               'C1_bar', 'C2_bar', 'I_bar', 'T_bar', 's', 's_bar', 'Fitness'] + list(metadata)

    if output_path.exists() and not overwrite_existing:
        with output_path.open(newline='') as existing_file:
            reader = csv.DictReader(existing_file)
            existing_rows = list(reader)
            compatible = reader.fieldnames == columns and len(existing_rows) == num_steps
        if compatible:
            try:
                compatible = (
                    np.array_equal([float(row['t']) for row in existing_rows], t)
                    and np.array_equal([float(row['tau']) for row in existing_rows], tau)
                    and np.array_equal([float(existing_rows[0][key]) for key in ('C1', 'C2', 'I', 'T')],
                                       [C1[0], C2[0], I[0], T[0]])
                    and all(row[key] == str(value)
                            for row in existing_rows for key, value in metadata.items())
                )
            except (TypeError, ValueError, KeyError):
                compatible = False
        if compatible:
            print("Skipping existing result:", output_path.name)
            continue
        raise ValueError(f"{output_path} uses different settings or is incomplete. "
                         "Choose another output_dir or set overwrite_existing=True to regenerate it.")

    # Genetic Algorithm setup

    # Arrays to hold fitness values
    fitness = np.zeros(num_steps)
    fitness_history = []

    # Create instance of GeneticAlgorithmDixon 
    ga_func = GeneticAlgorithmDixon()

    # Create an instance of the GA class
    ga_instance = pygad.GA(
        num_generations=30,
        num_parents_mating=10,
        sol_per_pop=100,
        num_genes=5,
        fitness_func=GeneticAlgorithmDixon.fitness_func_batch,
        fitness_batch_size=100,  # Evaluate Dixon candidates as a batch.
        on_generation=GeneticAlgorithmDixon.on_generation,
        keep_parents=-1,
        save_solutions=False,
        init_range_low=0.0,
        init_range_high=1.0,
        gene_space=[{'low': -10.0, 'high': 10.0}]*5,
        mutation_num_genes=2)

    print("Genetic Algorithm initialized.")
    integrator = dixon_integrate()
    if log_cell_integration:
        log_state = integrator.to_log_state([C1[0], C2[0], I[0], T[0]])

    for step in tqdm.tqdm(range(1, num_steps), desc=f"alpha={alpha:.6g}", leave=False):
        # Optimize the single IL-2 input for the current Dixon state.
        ga_instance.environment = {
            't': tau[step-1],
            'C1': C1[step-1], 'C2': C2[step-1],
            'I': I[step-1], 'T': T[step-1],
            'model_params': model_params,
            't_step': tau[step] - tau[step-1],
        }
        # Re-evaluate retained candidates when the environment changes.
        ga_instance.last_generation_parents = None
        ga_instance.last_generation_elitism = None
        ga_instance.run()
        best_solution, best_solution_fitness, _ = ga_instance.best_solution(
            pop_fitness=ga_instance.last_generation_fitness)
        fitness_history.append(best_solution_fitness)

        # Five genes ordered [C1, C2, I, T, 1]; clip negative treatment to zero.
        s_array[step] = GeneticAlgorithmDixon.solution_to_inputs(
            best_solution, C1[step-1], C2[step-1], I[step-1], T[step-1])

        params = [s_array[step]] + [model_params[key] for key in
            ('beta', 'mu_0', 'k1', 'mu_1', 'k2', 'mu_2', 'sigma', 'g', 'alpha')]

        # Advance the four Dixon states with this treatment held constant.
        state = log_state if log_cell_integration else [C1[step-1], C2[step-1], I[step-1], T[step-1]]
        next_state = integrator.integrate(
            state, params,
            t_span=(tau[step-1], tau[step]),
            method=integration_method,
            log_cells=log_cell_integration,
        )
        if log_cell_integration:
            log_state = next_state
            next_state = integrator.from_log_state(log_state)
        C1[step], C2[step], I[step], T[step] = next_state

        C1_bar[step] = C1[step] * K
        C2_bar[step] = C2[step] * K
        I_bar[step] = I[step] * K
        T_bar[step] = T[step] * K

    # Restore the physical IL-2 source rate (IU/day).
    s_bar = s_array * K * t_s


    with temp_output_path.open(mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(columns)
        for i in range(num_steps):
            writer.writerow([
                t[i], tau[i], C1[i], C2[i], I[i], T[i],
                C1_bar[i], C2_bar[i], I_bar[i], T_bar[i], s_array[i], s_bar[i],
                fitness_history[i-1] if i > 0 else '',
                *metadata.values(),
            ])
    temp_output_path.replace(output_path)
    print("Saved as", output_path)

print("Simulation completed.")
end_time = time.time()
print(f"Total simulation time: {(end_time - start_time)/60} minutes")


Dixon GA alpha sweep:   0%|          | 0/203 [00:00<?, ?it/s]

Genetic Algorithm initialized.


Dixon GA alpha sweep:   0%|          | 0/203 [00:34<?, ?it/s]


KeyboardInterrupt: 